# 02 — SQL Analysis

Load the cleaned, labeled dataset into a local SQLite database and answer 
business-style questions about churn/risk patterns using SQL rather than 
pandas — demonstrating the same analysis skill in the query language 
employers actually ask about in interviews.

In [8]:
import pandas as pd
import sqlite3

df = pd.read_csv('../data/clean_labeled_repos.csv')
print(df.shape)

conn = sqlite3.connect('../data/oss_health.db')
df.to_sql('repos', conn, if_exists='replace', index=False)

# confirm it landed correctly
pd.read_sql('SELECT COUNT(*) as row_count FROM repos', conn)

(498, 25)


,row_count
0,498


In [3]:
query = """
SELECT 
    star_tier,
    COUNT(*) as total_repos,
    SUM(CASE WHEN risk_category = 'High' THEN 1 ELSE 0 END) as high_risk_count,
    ROUND(100.0 * SUM(CASE WHEN risk_category = 'High' THEN 1 ELSE 0 END) / COUNT(*), 1) as high_risk_pct
FROM repos
GROUP BY star_tier
ORDER BY 
    CASE star_tier
        WHEN 'mega' THEN 1
        WHEN 'large' THEN 2
        WHEN 'mid' THEN 3
        WHEN 'small' THEN 4
        WHEN 'tiny' THEN 5
    END
"""
pd.read_sql(query, conn)

,star_tier,total_repos,high_risk_count,high_risk_pct
0,mega,100,3,3.0
1,large,100,8,8.0
2,mid,99,31,31.3
3,small,100,53,53.0
4,tiny,99,67,67.7


In [4]:
query = """
SELECT 
    CASE WHEN license = 'No license' THEN 'No license' ELSE 'Has license' END as license_status,
    COUNT(*) as total_repos,
    ROUND(100.0 * SUM(CASE WHEN risk_category = 'High' THEN 1 ELSE 0 END) / COUNT(*), 1) as high_risk_pct,
    ROUND(AVG(stars), 0) as avg_stars
FROM repos
GROUP BY license_status
"""
pd.read_sql(query, conn)

,license_status,total_repos,high_risk_pct,avg_stars
0,Has license,418,27.3,53396.0
1,No license,80,60.0,33106.0


In [5]:
query = """
SELECT 
    CASE 
        WHEN distinct_recent_commit_authors = 0 THEN '0 (no recent commits)'
        WHEN distinct_recent_commit_authors = 1 THEN '1 (single maintainer)'
        WHEN distinct_recent_commit_authors BETWEEN 2 AND 5 THEN '2-5'
        WHEN distinct_recent_commit_authors BETWEEN 6 AND 15 THEN '6-15'
        ELSE '16+'
    END as contributor_bucket,
    COUNT(*) as total_repos,
    ROUND(AVG(risk_score), 1) as avg_risk_score,
    ROUND(100.0 * SUM(CASE WHEN risk_category = 'High' THEN 1 ELSE 0 END) / COUNT(*), 1) as high_risk_pct
FROM repos
GROUP BY contributor_bucket
ORDER BY MIN(distinct_recent_commit_authors)
"""
pd.read_sql(query, conn)

,contributor_bucket,total_repos,avg_risk_score,high_risk_pct
0,0 (no recent commits),7,71.4,57.1
1,1 (single maintainer),63,83.7,77.8
2,2-5,123,40.5,39.8
3,6-15,142,31.7,26.8
4,16+,163,16.6,13.5


In [6]:
query = """
SELECT repo, stars, days_since_last_commit, commits_per_month_90d, is_archived, risk_score
FROM repos
WHERE distinct_recent_commit_authors = 0
ORDER BY risk_score DESC
"""
pd.read_sql(query, conn)

,repo,stars,days_since_last_commit,commits_per_month_90d,is_archived,risk_score
0,baidu/DDParser,999,1305.0,0.00,0,100
1,dogfun/chanlun,100,1678.0,0.00,0,100
2,evilfer/react-phaser,100,3785.0,0.00,0,100
3,StarfireLab/AutoZerologon,100,1078.0,0.00,0,100
4,gedoor/legado,47057,98.0,0.00,0,50
5,awesome-selfhosted/awesome-selfhosted,316666,0.0,30.67,0,25
6,piotrnar/gocoin,1000,3.0,22.33,0,25


In [9]:
query = """
SELECT COUNT(*) as affected_repos
FROM repos
WHERE distinct_recent_commit_authors <= 1 AND commits_per_month_90d >= 5
"""
pd.read_sql(query, conn)

,affected_repos
0,7


In [11]:
df = pd.read_csv('../data/clean_labeled_repos.csv')
df.to_sql('repos', conn, if_exists='replace', index=False)

query = """
SELECT COUNT(*) as affected_repos
FROM repos
WHERE distinct_recent_commit_authors <= 1 AND commits_per_month_90d >= 5
"""
pd.read_sql(query, conn)

,affected_repos
0,7


In [12]:
query = """
SELECT repo, stars, commits_per_month_90d, distinct_recent_commit_authors, 
       days_since_last_commit, risk_score, risk_category
FROM repos
WHERE distinct_recent_commit_authors <= 1 AND commits_per_month_90d >= 5
ORDER BY commits_per_month_90d DESC
"""
pd.read_sql(query, conn)

,repo,stars,commits_per_month_90d,distinct_recent_commit_authors,days_since_last_commit,risk_score,risk_category
0,Humanizr/Humanizer,9844,95.33,1,27.0,0,Low
1,AnInsomniacy/motrix-next,9922,62.33,1,3.0,0,Low
2,coreyhaines31/marketingskills,46568,53.33,1,0.0,0,Low
3,kavinsood/yaos,1000,33.67,1,5.0,0,Low
4,awesome-selfhosted/awesome-selfhosted,316666,30.67,0,0.0,0,Low
5,jeecgboot/JeecgBoot,47601,28.33,1,1.0,0,Low
6,piotrnar/gocoin,1000,22.33,0,3.0,0,Low


In [13]:
query = """
SELECT 
    language,
    COUNT(*) as total_repos,
    ROUND(AVG(risk_score), 1) as avg_risk_score,
    ROUND(100.0 * SUM(CASE WHEN risk_category = 'High' THEN 1 ELSE 0 END) / COUNT(*), 1) as high_risk_pct
FROM repos
WHERE language != 'None/Not detected'
GROUP BY language
HAVING total_repos >= 10
ORDER BY avg_risk_score DESC
"""
pd.read_sql(query, conn)

,language,total_repos,avg_risk_score,high_risk_pct
0,Java,17,56.5,58.8
1,C#,12,53.3,50.0
2,JavaScript,68,43.2,45.6
3,C,16,40.9,31.3
4,HTML,11,38.2,36.4
5,Shell,15,36.0,40.0
6,Python,92,34.1,33.7
7,TypeScript,60,21.0,16.7
8,C++,24,18.5,16.7
9,Go,29,15.9,13.8


In [14]:
query = """
SELECT 
    risk_category,
    ROUND(AVG(open_issues_total), 1) as avg_open_issues,
    ROUND(AVG(open_issues_total * 1.0 / (commits_per_month_90d + 0.1)), 1) as avg_issues_per_commit_activity
FROM repos
GROUP BY risk_category
ORDER BY 
    CASE risk_category WHEN 'High' THEN 1 WHEN 'Medium' THEN 2 WHEN 'Low' THEN 3 END
"""
pd.read_sql(query, conn)

,risk_category,avg_open_issues,avg_issues_per_commit_activity
0,High,54.6,545.7
1,Medium,513.1,5047.6
2,Low,864.9,41.1


In [15]:
query = """
SELECT repo, risk_category, open_issues_total, commits_per_month_90d,
       ROUND(open_issues_total * 1.0 / (commits_per_month_90d + 0.1), 1) as issues_per_activity
FROM repos
ORDER BY issues_per_activity DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,repo,risk_category,open_issues_total,commits_per_month_90d,issues_per_activity
0,type-challenges/type-challenges,Medium,33198,0.0,331980.0
1,ytdl-org/youtube-dl,Medium,3627,0.0,36270.0
2,AUTOMATIC1111/stable-diffusion-webui,High,2421,0.0,24210.0
3,Aider-AI/aider,Medium,1360,0.0,13600.0
4,GitHubDaily/GitHubDaily,Medium,794,0.0,7940.0
5,pnp/PnP-PowerShell,High,778,0.0,7780.0
6,open-mmlab/mmsegmentation,High,773,0.0,7730.0
7,Theano/Theano,High,596,0.0,5960.0
8,slab/quill,Medium,578,0.0,5780.0
9,evolus/pencil,Medium,484,0.0,4840.0


In [16]:
df['issues_per_activity'] = df['open_issues_total'] / (df['commits_per_month_90d'] + 0.1)

summary = df.groupby('risk_category').agg(
    median_open_issues=('open_issues_total', 'median'),
    median_issues_per_activity=('issues_per_activity', 'median')
).reindex(['High', 'Medium', 'Low'])

summary

,median_open_issues,median_issues_per_activity
risk_category,,
High,6.0,60.000000
Medium,15.0,130.000000
Low,89.5,2.944419


## SQL Analysis Summary

| Question | Finding |
|---|---|
| Does star tier predict risk? | Yes, strongly and monotonically: 3% high-risk (mega) → 68% (tiny) |
| Does license presence predict risk? | Yes: no-license repos are 2x as likely to be high-risk (60% vs 27.3%) |
| Does contributor count predict risk? | Yes, but the API undercounts contributors when commit emails aren't linked to a GitHub account — found and fixed a labeling bug affecting 7/498 repos (1.4%) as a result |
| Does language predict risk? | Correlated but confounded — likely reflects project generation/culture (Rust/Go skew newer and more actively championed) more than language itself causing risk |
| Does open issue count predict risk? | No — raw counts are backwards (Low-risk repos have the *most* open issues, since dead projects stop attracting new issue filers). Issues-per-activity is a more nuanced signal, peaking in the Medium-risk band rather than High. |

**Data quality note:** an initial mean-based version of the issues-per-activity 
analysis was distorted by extreme outliers (one repo with 33k+ open issues 
and zero recent commits); switched to median partway through for a more 
representative picture.